In [ ]:
import pandas as pd
import numpy as np
import re
import random

from spacy.lang.ru import Russian

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchtext.data import Field

# Загрузка данных, проверка дубликатов и пустых строк

In [6]:
# Загружаем tsv-файл с диалогами (разделитель табуляция)
df = pd.read_csv("/Users/albertzagibin/Documents/University/ML/7 sem/Seq2Seq/dialogues.tsv", sep="\t")
df["dialogue"][1]

'<span class=participant_1>Пользователь 1: Привет!</span><br /><span class=participant_2>Пользователь 2: Привет,Как жизнь?</span><br /><span class=participant_1>Пользователь 1: Отлично) Солнышко светит, птички поют!</span><br /><span class=participant_2>Пользователь 2: Я вот сегодня понял, что меня тупо используют, всем<br />нужны от меня лишь деньги, ненавижу людей</span><br /><span class=participant_2>Пользователь 2: Чем занимаешься по жизни, я вот бизнесмен.</span><br /><span class=participant_1>Пользователь 1: А я вот учу детей, работаю с начальными классами</span><br /><span class=participant_1>Пользователь 1: Не все люди такие, как ты говоришь</span><br /><span class=participant_1>Пользователь 1: Помимо работы чем еще ты занимаешься?</span><br /><span class=participant_2>Пользователь 2: К свадьбе готовлюсь</span><br /><span class=participant_2>Пользователь 2: А ты?</span><br /><span class=participant_1>Пользователь 1: Вот видишь) значит, нашел такую женщину, которой не<br />нужны

In [7]:
df.columns

Index(['persona_1_profile', 'persona_2_profile', 'dialogue'], dtype='object')

In [8]:
# Оставляем только колонку с диалогами
df = df[["dialogue"]]
df.head(3)

,dialogue
0,<span class=participant_2>Пользователь 2: Прив...
1,<span class=participant_1>Пользователь 1: Прив...
2,<span class=participant_1>Пользователь 1: Прив...


In [9]:
num_duplicates = df.duplicated().sum()
print(f"Количество дублирующихся строк: {num_duplicates}")
print(f"Всего строк в датасете: {len(df)}")

Количество дублирующихся строк: 202
Всего строк в датасете: 10013


In [10]:
df = df.drop_duplicates(subset=["dialogue"])

In [11]:
# Приводим колонку dialogue к строковому типу
df["dialogue"] = df["dialogue"].astype(str)


In [12]:
num_duplicates = df.duplicated().sum()
print(f"Количество дублирующихся строк: {num_duplicates}")

Количество дублирующихся строк: 0


# Предобработка текста


Первым делом, проанализировав структуру диалога в Датасете, наобходимо выполнить следующие шаги

1.   убрать HTML‑теги
2.   сохранить номер говорящего
3.   получить понятные реплики.


======================================================================


Проанализируем тэги:
1.  `<span class=participant_2>`  
можно извлечь из этого тэга, кто говорит: участник 1 или 2


Как анализировать такие строки  
1. Реплика всегда находится внутри  
`"<span class=participant_X> ... </span>".`  
Значит, первая задача — выделить все такие фрагменты и вытащить X — номер говорящего;

======================================================================

Я не смотрела каждую строчку, но предполагаю на основе первых двух следующее:  
* `<br />` внутри `<span>` означает просто перенос строки в рамках одного сообщения  
* `<br />` снаружи (между `</span><br /><span...>`): это просто разделитель реплик.

======================================================================

Префикс "Пользователь 1:" / "Пользователь 2:" внутри текста дублирует информацию о говорящем. Поэтому он нам не потребуется. Тут снова на основе первых строк предполагаю, что везде в каждой строке используется именно такая структура


## Функция для обработки HTML - тегов

In [13]:
found_parts = re.findall(r'<span\s+class=participant_(\d+)>(.*?)</span>', df["dialogue"][0], re.DOTALL)
found_parts

[('2', 'Пользователь 2: Привет) расскажи о себе'),
 ('1',
  'Пользователь 1: Привет) под вкусный кофеек настроение поболтать появилось<br />)'),
 ('2', 'Пользователь 2: Что читаешь? Мне нравится классика'),
 ('2', 'Пользователь 2: Я тоже люблю пообщаться'),
 ('1', 'Пользователь 1: Люблю животных, просто обожаю, как и свою работу)'),
 ('1', 'Пользователь 1: Я фантастику люблю'),
 ('2', 'Пользователь 2: А я выращиваю фиалки'),
 ('2', 'Пользователь 2: И веду здоровый и активный образ жизни!'),
 ('1', 'Пользователь 1: Ух ты, интересно.'),
 ('2',
  'Пользователь 2: Ты случайно не принц на белом коне? Я его очень жду<br />..'),
 ('1',
  'Пользователь 1: А у меня из хобби каждую неделю тусить с моим лучшим<br />другом)')]

In [14]:
'Пользователь 2: Привет) расскажи о себе: кто ты по жизни'.split(':')

['Пользователь 2', ' Привет) расскажи о себе', ' кто ты по жизни']

In [15]:
def create_dataset(dialog):
    """
    Преобразует HTML-диалог в список (speaker_id, text).
    Обрабатываем:
      - <span class=participant_1> и participant_2
      - <br /> как разделитель внутри реплики
      - префиксы "Пользователь 1/2:"
    """
    # Если это не текст (например, NaN), сразу выходим
    if not isinstance(dialog, str):
        return []

    result_list = []  #  пары: ('1', 'текст'), ('2', 'текст')

    # Делим весь HTML на части по тегу <span...>
    found_parts = re.findall(r'<span\s+class=participant_(\d+)>(.*?)</span>', dialog, re.DOTALL)


    for speaker_number, text in found_parts:

        # Заменяем <br /> на обычный пробел
        message = text.replace('<br />', ' ')

        # Удаляем пользователя
        if message.startswith('Пользователь 1:') or message.startswith('Пользователь 2:'):
            message = message.split(':', 1)[1]

        # Удаляем любые остатки HTML-тегов
        message = re.sub(r'<.*?>', '', message)

        # Убираем лишние пробелы
        message = ' '.join(message.split())

        if message != '':
            result_list.append((speaker_number, message))

    return result_list

In [16]:
df["clean"] = df["dialogue"].apply(create_dataset)

In [17]:
df['clean'].iloc[1]

[('1', 'Привет!'),
 ('2', 'Привет,Как жизнь?'),
 ('1', 'Отлично) Солнышко светит, птички поют!'),
 ('2',
  'Я вот сегодня понял, что меня тупо используют, всем нужны от меня лишь деньги, ненавижу людей'),
 ('2', 'Чем занимаешься по жизни, я вот бизнесмен.'),
 ('1', 'А я вот учу детей, работаю с начальными классами'),
 ('1', 'Не все люди такие, как ты говоришь'),
 ('1', 'Помимо работы чем еще ты занимаешься?'),
 ('2', 'К свадьбе готовлюсь'),
 ('2', 'А ты?'),
 ('1',
  'Вот видишь) значит, нашел такую женщину, которой не нужны от тебя деньги'),
 ('2', 'Да я надеюсь на это,люблю ее')]

# Создание диалогового датасета

Промежуточное представление Диалога

In [18]:
df['clean'][0]

[('2', 'Привет) расскажи о себе'),
 ('1', 'Привет) под вкусный кофеек настроение поболтать появилось )'),
 ('2', 'Что читаешь? Мне нравится классика'),
 ('2', 'Я тоже люблю пообщаться'),
 ('1', 'Люблю животных, просто обожаю, как и свою работу)'),
 ('1', 'Я фантастику люблю'),
 ('2', 'А я выращиваю фиалки'),
 ('2', 'И веду здоровый и активный образ жизни!'),
 ('1', 'Ух ты, интересно.'),
 ('2', 'Ты случайно не принц на белом коне? Я его очень жду ..'),
 ('1', 'А у меня из хобби каждую неделю тусить с моим лучшим другом)')]

In [19]:
rows = []
for idx, text in enumerate(df["clean"]):
    for number_message, (speaker, message) in enumerate(text):
        rows.append({
            "id_dialog": idx, # номер диалога
            "number_message": number_message, # номер реплики в диалоге
            "speaker": speaker, # участник
            "text": message # текст
        })

# Собираем в DataFrame
dialog_df = pd.DataFrame(rows)
dialog_df.to_csv("dialog_df.csv", index=False, encoding="utf-8")
print("Строк в dialog_steps_df:", len(dialog_df))
pd.set_option('display.max_colwidth', 120)
display(dialog_df.head(5))


Строк в dialog_steps_df: 255004


,id_dialog,number_message,speaker,text
0,0,0,2,Привет) расскажи о себе
1,0,1,1,Привет) под вкусный кофеек настроение поболтать появилось )
2,0,2,2,Что читаешь? Мне нравится классика
3,0,3,2,Я тоже люблю пообщаться
4,0,4,1,"Люблю животных, просто обожаю, как и свою работу)"


**Вариант 1:**  
Каждая реплика в диалоге считается "входом", а следующая за ней — "ответом".  
то есть диалог будет примерно строится по такому принципу:  
реплика 1 -> реплика 2  
реплика 2 -> реплика 3 и тд

In [20]:
# Создаём два списка: для "входов" и "ответов"
prev_list, next_list = [], []

# Проходим по всем диалогам
for data in df["clean"]:
    # Пропускаем диалоги короче двух реплик
    if len(data) < 2:
        continue
    # Берём каждую реплику и следующую за ней
    for i in range(len(data) - 1):
        _, text_prev = data[i]      # текущая реплика
        _, text_next = data[i + 1]  # следующая реплика

        prev_list.append(text_prev)
        next_list.append(text_next)

# Собираем в DataFrame
pairs_df = pd.DataFrame({"prev_text": prev_list, "next_text": next_list})
pairs_df.to_csv("pairs_df.csv", index=False, encoding="utf-8")

print("Строк в pairs_any_df:", len(pairs_df))
pd.set_option('display.max_colwidth', 120)
display(pairs_df.head(5))

Строк в pairs_any_df: 245193


,prev_text,next_text
0,Привет) расскажи о себе,Привет) под вкусный кофеек настроение поболтать появилось )
1,Привет) под вкусный кофеек настроение поболтать появилось ),Что читаешь? Мне нравится классика
2,Что читаешь? Мне нравится классика,Я тоже люблю пообщаться
3,Я тоже люблю пообщаться,"Люблю животных, просто обожаю, как и свою работу)"
4,"Люблю животных, просто обожаю, как и свою работу)",Я фантастику люблю


Преимущества такого подхода:  
* Простота реализации  
* Большое количество данных  

Недостатки:  
* Нарушение диалоговой логики
* Нет понимания смены говорящего

**Вариант 2:**  
каждая пара — это реплика от участника 1 и ответ от участника 2.


In [21]:
user1_list = []
user2_list = []

for data in df["clean"]:
    if len(data) < 2:
        continue
    for i in range(len(data) - 1):
        sp_cur, txt_cur = data[i]
        sp_next, txt_next = data[i + 1]

        if sp_cur == "1" and sp_next == "2": # либо if sp_cur != sp_next:
            user1_list.append(txt_cur)
            user2_list.append(txt_next)

# Собираем в DataFrame
pairs_12_df = pd.DataFrame({"user_1": user1_list, "user_2": user2_list})
pairs_12_df.to_csv("pairs_12_df.csv", index=False, encoding="utf-8")

print("Строк в pairs_any_df:", len(pairs_12_df))
pd.set_option('display.max_colwidth', 120)
display(pairs_12_df.head(5))

Строк в pairs_any_df: 77944


,user_1,user_2
0,Привет) под вкусный кофеек настроение поболтать появилось ),Что читаешь? Мне нравится классика
1,Я фантастику люблю,А я выращиваю фиалки
2,"Ух ты, интересно.",Ты случайно не принц на белом коне? Я его очень жду ..
3,Привет!,"Привет,Как жизнь?"
4,"Отлично) Солнышко светит, птички поют!","Я вот сегодня понял, что меня тупо используют, всем нужны от меня лишь деньги, ненавижу людей"


Преимущества такого подхода:  
* Соблюдается диалоговая логика
* Большое количество данных  

Недостатки:  
* Потеря почти половины данных
* Нет понимания смены говорящего
* Игнорирование диалогов, начатых участником 2

**Вариант 3:**  
каждая пара — это реплика от участника 1 и ответ от участника 2.  
С улучшенной версией - обьединение подряд идущих реплик от одного участника диалога

In [22]:
def merge_replic(data):
    merged = []
    cur_speaker, cur_text = data[0] # первая реплика в первой строке
    # Проходим по остальным репликам
    for speaker, text in data[1:]:
        if speaker == cur_speaker:
            cur_text += ". " + text
        else:
            merged.append((cur_speaker, cur_text))
            cur_speaker, cur_text = speaker, text

    merged.append((cur_speaker, cur_text))
    return merged


# Список всех пар
dialog_pairs = []
for data in df["clean"]:
    # Объединяем подряд идущие реплики одного участника
    merged_message = merge_replic(data)

    for i in range(len(merged_message) - 1):
        input_text = merged_message[i][1]
        output_text = merged_message[i + 1][1]
        dialog_pairs.append((input_text, output_text))

In [23]:
# Преобразуем список пар в таблицу
final_df = pd.DataFrame(dialog_pairs, columns=["input", "output"])
# Сохраняем в файл
final_df.to_csv("final_df.csv", index=False, encoding="utf-8")
# Выводим статистику
print(f"В датасете {len(final_df)} обучающих пар.")
pd.set_option('display.max_colwidth', 150)
display(final_df.head(10))

В датасете 155686 обучающих пар.


,input,output
0,Привет) расскажи о себе,Привет) под вкусный кофеек настроение поболтать появилось )
1,Привет) под вкусный кофеек настроение поболтать появилось ),Что читаешь? Мне нравится классика. Я тоже люблю пообщаться
2,Что читаешь? Мне нравится классика. Я тоже люблю пообщаться,"Люблю животных, просто обожаю, как и свою работу). Я фантастику люблю"
3,"Люблю животных, просто обожаю, как и свою работу). Я фантастику люблю",А я выращиваю фиалки. И веду здоровый и активный образ жизни!
4,А я выращиваю фиалки. И веду здоровый и активный образ жизни!,"Ух ты, интересно."
5,"Ух ты, интересно.",Ты случайно не принц на белом коне? Я его очень жду ..
6,Ты случайно не принц на белом коне? Я его очень жду ..,А у меня из хобби каждую неделю тусить с моим лучшим другом)
7,Привет!,"Привет,Как жизнь?"
8,"Привет,Как жизнь?","Отлично) Солнышко светит, птички поют!"
9,"Отлично) Солнышко светит, птички поют!","Я вот сегодня понял, что меня тупо используют, всем нужны от меня лишь деньги, ненавижу людей. Чем занимаешься по жизни, я вот бизнесмен."


Проверка на дубликаты и пустые значения

In [24]:
print(final_df.isna().sum())

input     0
output    0
dtype: int64


In [25]:
print("Всего строк:", len(final_df))
num_duplicates = final_df.duplicated().sum()
print(f"Количество дублирующихся строк: {num_duplicates}")
# Сколько ровно дубликатов
n_dups = len(final_df) - num_duplicates
print("Дубликатов:", n_dups)


Всего строк: 155686
Количество дублирующихся строк: 2857
Дубликатов: 152829


In [26]:
final_df = final_df.drop_duplicates().reset_index(drop=True)
print(f"В датасете {len(final_df)} обучающих пар.")

В датасете 152829 обучающих пар.


# NLP

In [2]:
final_df = pd.read_csv('final_df.csv')

In [3]:
final_df.head()

,input,output
0,Привет) расскажи о себе,Привет) под вкусный кофеек настроение поболтат...
1,Привет) под вкусный кофеек настроение поболтат...,Что читаешь? Мне нравится классика. Я тоже люб...
2,Что читаешь? Мне нравится классика. Я тоже люб...,"Люблю животных, просто обожаю, как и свою рабо..."
3,"Люблю животных, просто обожаю, как и свою рабо...",А я выращиваю фиалки. И веду здоровый и активн...
4,А я выращиваю фиалки. И веду здоровый и активн...,"Ух ты, интересно."


In [4]:
spacy_ru = Russian()

def tokenize_ru(text):
    return[tok.text for tok in spacy_ru(text)]

In [5]:
SRC = Field(tokenize=tokenize_ru,
    lower=True,
    init_token='<sos>',
    eos_token='<eos>',
    pad_token='<pad>',
    unk_token='<unk>'
)

TRG = Field(tokenize=tokenize_ru,
    lower=True,
    init_token='<sos>',
    eos_token='<eos>',
    pad_token='<pad>',
    unk_token='<unk>'
)

In [6]:
inputs = final_df['input'].tolist()
outputs = final_df['output'].tolist()

inputs_tokens = [tokenize_ru(text) for text in inputs]
output_tokens_trg = [tokenize_ru(text) for text in outputs]

SRC.build_vocab(inputs_tokens, min_freq=2, max_size=10000)
TRG.build_vocab(output_tokens_trg, min_freq=2, max_size=10000)


In [8]:
def numericalize(tokens, vocab):
    return [vocab.stoi.get(tok, vocab.stoi['<unk>']) for tok in tokens]

In [9]:
class Seq2SeqDataset(Dataset):
    def __init__(self, df, src_field, trg_field):
        self.df = df.reset_index(drop=True)
        self.src_field = src_field
        self.trg_field = trg_field

    def __len__(self):
        return len(self.df)
    
    def encode(self, text, field):
        tokens = field.tokenize(text)
        tokens = [field.init_token] + tokens + [field.eos_token]
        return numericalize(tokens, field.vocab)

    def __getitem__(self, idx):
        src_text = self.df.loc[idx, 'input']
        trg_text = self.df.loc[idx, 'output']

        src_ids = self.encode(src_text, self.src_field)
        trg_ids = self.encode(trg_text, self.trg_field)

        return {
            'src_ids': torch.tensor(src_ids, dtype=torch.long),
            'trg_ids': torch.tensor(trg_ids, dtype=torch.long)
        }

In [10]:
def collate_fn(batch):
    src_batch = [item['src_ids'] for item in batch]
    trg_batch = [item['trg_ids'] for item in batch]
    
    src_batch = pad_sequence(src_batch, padding_value=SRC.vocab.stoi['<pad>'])
    trg_batch = pad_sequence(trg_batch, padding_value=TRG.vocab.stoi['<pad>'])

    return src_batch, trg_batch

In [130]:
np.random.seed(60)
df_batch = np.random.choice(final_df.index, size=int(final_df.shape[0]/100), replace=False)

In [131]:
short_df = final_df.iloc[df_batch]
short_df.shape

(1556, 2)

In [132]:
dataset = Seq2SeqDataset(short_df, SRC, TRG)

batch_size = 8

loader = DataLoader(
    dataset,
    collate_fn=collate_fn, 
    batch_size=batch_size, 
    shuffle=True
)


In [104]:
src, trg = next(iter(loader))

print(src.shape)  # [src_len, batch_size]
print(trg.shape)  # [trg_len, batch_size]

torch.Size([34, 8])
torch.Size([36, 8])


In [133]:
sentence_idx = 2 # номер предложения в батче 

tokens = [SRC.vocab.itos[src[t, sentence_idx].item()] for t in range(src.size(0))] 

print(" ".join(tokens))

<sos> А я больше мечтаю о жизни .... Многие говорят , что со мной скучно . Это обидно немного ; ( . Где вы живёте ? <eos> <pad> <pad> <pad> <pad>


### Seq2Seq

In [24]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


device = get_device()
print(f"Using device: {device}")

Using device: mps


In [146]:
class Encoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
        super().__init__()
        self.dropout = nn.Dropout(p)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)

    def forward(self, x):
        # x shape: (seq_length, N) where N is batch size
        
        embedding = self.dropout(self.embedding(x))
        # embedding shape: (seq_length, N, embedding_size)

        _, (hidden, cell) = self.rnn(embedding)

        return hidden, cell
    
class Decoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, output_size, num_layers, p):
        super().__init__()
        self.dropout = nn.Dropout(p)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell):
        # x shape: (N) where N is batch size, we need (1, N) 
        # is 1 because we are sending in a single word

        x = x.unsqueeze(0)

        embedding = self.dropout(self.embedding(x))
        # embedding shape: (1, N, embedding_size)

        outputs, (hidden, cell) = self.rnn(embedding, (hidden, cell))
        # output shape: (1, N, hidden_size)

        predictions = self.fc(outputs)
        # predictions shape: (1, N, length_target_vocabulary) to send it to
        # loss function we want it to be (N, length_target_vocabulary) so we're
        # just gonna remove the first dim
        predictions = predictions.squeeze(0)

        return predictions, hidden, cell
    

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio=0.5):
        batch_size = source.shape[1]
        target_len = target.shape[0]
        target_vocab_size = len(TRG.vocab)

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(device)

        hidden,cell = self.encoder(source)
        
        # Grab the first input to the Decoder which will be <sos> token
        x = target[0]

        for t in range(1, target_len):
            output, hidden, cell = self.decoder(x, hidden, cell)

            # Store next output prediction
            outputs[t] = output

            # Get the best word the Decoder predicted (index in vocab)
            best_guess = output.argmax(1)

            # Teacher Forcing
            x = target[t] if random.random() < teacher_force_ratio else best_guess

        return outputs

In [249]:
# Training hyperparameters
num_epochs = 5
learning_rate = 0.001

# Model hyperparameters
load_model = False
input_size_encoder = len(SRC.vocab)
input_size_decoder = len(TRG.vocab)
output_size = len(TRG.vocab)
encoder_embedding_size = 300
decoder_embedding_size = 300
hidden_size = 1024  # Needs to be the same for both RNN's
num_layers = 2
enc_dropout = 0.37
dec_dropout = 0.37

In [27]:
encoder_net = Encoder(
    input_size_encoder,
    encoder_embedding_size,
    hidden_size,
    num_layers,
    enc_dropout
).to(device)

decoder_net = Decoder(
    input_size_decoder,
    decoder_embedding_size,
    hidden_size,
    output_size,
    num_layers,
    dec_dropout
).to(device)

In [28]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])


In [143]:
def top_k_sampling(output, k=5):
    topk_probs, topk_indices = torch.topk(output, k)
    probs = torch.softmax(topk_probs, dim=1)
    idx = torch.multinomial(probs, 1).item()
    return topk_indices[0, idx].item()


def answer_sentence(model, sentence, SRC, TRG, device, max_length=50):
    tokens = tokenize_ru(sentence) 
    tokens.insert(0, SRC.init_token)
    tokens.append(SRC.eos_token)
    tokens_ids = numericalize(tokens, SRC.vocab) 
    
    sentence_tensor = torch.LongTensor(tokens_ids).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(sentence_tensor)

    outputs = [TRG.vocab.stoi["<sos>"]]

    for _ in range(max_length):
        previous_word = torch.LongTensor([outputs[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(previous_word, hidden, cell)
            best_guess = top_k_sampling(output, k=5)
        
        outputs.append(best_guess)

        if best_guess == TRG.vocab.stoi["<eos>"]:
            break

    answer = [TRG.vocab.itos[idx] for idx in outputs]

    # remove start token
    answer_str = ' '.join(answer[1:])
    return answer_str.replace("<eos>", "")


In [148]:
model = Seq2Seq(encoder_net, decoder_net).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

pad_idx = TRG.vocab.stoi["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)


In [ ]:
if load_model:
    load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)

In [31]:
sentence = "Привет. Я живу на дальнем востоке, а ты?"

In [250]:
for epoch in range(1, num_epochs+1):
    print(f"[Epoch {epoch} / {num_epochs}]")

    model.eval()
    
    with torch.no_grad():
        answer = answer_sentence(model, sentence, SRC, TRG, device, max_length=20)
    
    print(f'Answer example sentence: \n {answer}')

    model.train()

    total_loss = 0 

    for batch_idx, (src, trg) in enumerate(loader):
        inp_data = src.to(device)
        target = trg.to(device)

        output = model(inp_data, target, teacher_force_ratio=0.85)

        output = output[1:].reshape(-1, output.shape[2])
        target = target[1:].reshape(-1)

        optimizer.zero_grad()
        loss = criterion(output, target)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()
        total_loss += loss.item()

    print(f"Train Loss: {total_loss:.4f}")

    checkpoint = {"state_dict": model.state_dict(), "optimizer": optimizer.state_dict()}
    save_checkpoint(checkpoint)



[Epoch 1 / 5]
Answer example sentence: 
 Ого ) я всегда люблю кошек крестиком . А ты ? 
Train Loss: 455.9144
=> Saving checkpoint
[Epoch 2 / 5]
Answer example sentence: 
 Работаю я слесарем ) ) ) котёнка , <unk> . А недалеко <unk> умею играть . Лето очень хочу посетить
Train Loss: 444.8355
=> Saving checkpoint
[Epoch 3 / 5]
Answer example sentence: 
 Я молодая пока не получается . А ты посоветуешь деток ? 
Train Loss: 440.5772
=> Saving checkpoint
[Epoch 4 / 5]
Answer example sentence: 
 Я лично флорист , но очень хочу посетить юристом , так люблю ) ) 
Train Loss: 439.1680
=> Saving checkpoint
[Epoch 5 / 5]
Answer example sentence: 
 Я молодая . А ты ? ? Чем интересуешься вообще ? 
Train Loss: 438.9823
=> Saving checkpoint


In [264]:
s = 'Привет. Я живу на дальнем востоке, а ты?'

while s != 'quit':
    print(f'input: {s}') 
    with torch.no_grad():
        answer = answer_sentence(model, s, SRC, TRG, device, max_length=20)
    
    print(f'Валера: {answer}')
    s = input()

input: Привет. Я живу на дальнем востоке, а ты?
Валера: Я из городка . Люблю сладенькое 
input: кем работаешь?
Валера: А шить в будущем . А расскажи о себе , работаешь ? 
input: кем работаешь?]
Валера: Нет Санкт возраст , как- нибудь <unk> не люблю 
input: кем работаешь?
Валера: Собака 
input: кем работаешь?
Валера: Любим <unk> . Мне 25 . А ты кем работаешь ? 
input: кем работаешь?
Валера: Собака три собой с внуками <unk> 
input: кем работаешь?
Валера: <unk> . А вы ? 
input: кем работаешь?
Валера: А какой города ? 
input: где ты живешь
Валера: ты ? 
input: где ты живешь?
Валера: <unk> . У меня хорошо <unk> У тебя есть блог 
input: как дела
Валера: <unk> . А по из 


In [263]:
for _ in range(10):
    print(answer_sentence(model, s, SRC, TRG, device, max_length=50))

Очень приятно с тобой ) ) ) ) 
Ладно мне пора , приятно было приятно познакомиться 
я не так не с пацанами свои моделью и у меня высшее две автомобиль . Очень приятно было пообщаться 
Очень приятно с тобой познакомиться 
В <unk> . Но не не с друзьями не с кем то ) 
Очень хочу не <unk> рано заготовки и я не смогу и не с котом 
Только собаку не дает семьи мы с ним и мужчина ) 
Я с собой не с внуками дни 
Очень приятно с тобой познакомиться 
Я езжу на гитаре . Моя мечта 


In [153]:
short_df.head(10)

,input,output
147282,Здорово.,А кто Вы по профессии?
62131,На гонщика),Рэп- это интересно. Ритмичная музыка- под неё ...
39966,Белла),Красивое имя 😊👍
1280,Ок. Привет,Привет рад знакомству!
126780,Много операций сделал?,"Да, более 500 точно."
117864,"Ух ты здорово, я тоже люблю сладкое, но из гор...","Да пицца не плохо, мне нравится ее готовить"
76931,Что за судно у тебя?. Люблю читать про пиратов),Clamato. Camaro*. Возможно слыхал. О таком
95701,Да точно сказано,А ты чем в свободное время занимаешься?. Город...
10458,кем работаешь?,Госслужащий. А ты?
60886,Что тебе нравится читать?,да
